In [3]:
!pip install streamlit pyngrok --quiet
print("Done!")

Done!


In [4]:
import json, os

# Same kaggle setup as before
kaggle_config = {
    "username": "your_kaggle_username",
    "key": "your_kaggle_token"
}

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_config, f)
os.chmod('/root/.kaggle/kaggle.json', 600)

print("Kaggle setup done!")

Kaggle setup done!


In [5]:
!kaggle datasets download -d clmentbisaillon/fake-and-real-news-dataset
!unzip fake-and-real-news-dataset.zip -d fake_news_data
print("Dataset ready!")

Dataset URL: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
License(s): CC-BY-NC-SA-4.0
fake-and-real-news-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  fake-and-real-news-dataset.zip
replace fake_news_data/Fake.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace fake_news_data/True.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: N
Dataset ready!


In [6]:
import pandas as pd

fake = pd.read_csv('/content/fake_news_data/Fake.csv')
real = pd.read_csv('/content/fake_news_data/True.csv')

print(f"Fake news articles: {len(fake)}")
print(f"Real news articles: {len(real)}")
print(f"\nFake sample: {fake['title'][0]}")
print(f"\nReal sample: {real['title'][0]}")

Fake news articles: 23481
Real news articles: 21417

Fake sample:  Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing

Real sample: As U.S. budget fight looms, Republicans flip their fiscal script


In [7]:
import pandas as pd
import numpy as np

# Add labels
fake['label'] = 0  # 0 = Fake
real['label'] = 1  # 1 = Real

# Combine datasets
df = pd.concat([fake, real], ignore_index=True)

# Use title + text combined
df['content'] = df['title'] + " " + df['text']
df = df[['content', 'label']].dropna()

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total articles: {len(df)}")
print(f"Fake: {sum(df.label == 0)} | Real: {sum(df.label == 1)}")

Total articles: 44898
Fake: 23481 | Real: 21417


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['content'], df['label'], test_size=0.2, random_state=42)

# TF-IDF Vectorizer
print("Vectorizing text...")
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Train model
print("Training model...")
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"\n✅ Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))

Vectorizing text...
Training model...

✅ Accuracy: 98.78%

Classification Report:
              precision    recall  f1-score   support

        Fake       0.99      0.99      0.99      4710
        Real       0.98      0.99      0.99      4270

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [9]:
import pickle

with open('fake_news_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print("Model saved!")

Model saved!


In [10]:
app_code = '''
import streamlit as st
import pickle
import numpy as np

st.set_page_config(page_title="Fake News Detector", page_icon="📰", layout="centered")

@st.cache_resource
def load_model():
    with open("fake_news_model.pkl", "rb") as f:
        model = pickle.load(f)
    with open("tfidf_vectorizer.pkl", "rb") as f:
        tfidf = pickle.load(f)
    return model, tfidf

model, tfidf = load_model()

st.title("📰 Fake News Detector")
st.markdown("Enter a news headline or article to check if it is **Real or Fake!**")
st.divider()

text_input = st.text_area("Enter news headline or article:",
                           placeholder="Paste any news headline or article here...",
                           height=150)

if st.button("Check News 🔍"):
    if text_input.strip() == "":
        st.warning("Please enter some text!")
    else:
        with st.spinner("Analyzing..."):
            input_tfidf = tfidf.transform([text_input])
            prediction = model.predict(input_tfidf)[0]
            probability = model.predict_proba(input_tfidf)[0]

        if prediction == 1:
            st.success(f"✅ This news appears to be **REAL** with {probability[1]*100:.2f}% confidence!")
        else:
            st.error(f"🚨 This news appears to be **FAKE** with {probability[0]*100:.2f}% confidence!")

        st.divider()
        col1, col2 = st.columns(2)
        with col1:
            st.metric("🟢 Real Probability", f"{probability[1]*100:.2f}%")
        with col2:
            st.metric("🔴 Fake Probability", f"{probability[0]*100:.2f}%")

st.divider()
st.markdown("**Try these examples:**")
col1, col2 = st.columns(2)
with col1:
    if st.button("📰 Real News Example"):
        st.info("As U.S. budget fight looms, Republicans flip their fiscal script")
with col2:
    if st.button("🚨 Fake News Example"):
        st.info("Donald Trump Sends Out Embarrassing New Year Eve Message This is Disturbing")

st.caption("Built by N. Priya Dharshini | PhD Research Scholar | Kalasalingam University")
'''

with open('fake_news_app.py', 'w') as f:
    f.write(app_code)

print("App created!")

App created!


In [20]:

from pyngrok import ngrok
import subprocess, time

ngrok.set_auth_token("your_token_here")

process = subprocess.Popen(["streamlit", "run", "/content/fake_news_app.py",
                            "--server.port=8501", "--server.headless=true"])
time.sleep(12)

public_url = ngrok.connect(8501)
print(f"\n✅ Fake News app live at: {public_url}")

PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://democrat-skewed-storewide.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}


In [21]:
!pip install gradio --quiet
print("Done!")

Done!


In [22]:
import gradio as gr
import pickle

# Load models
with open('fake_news_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('tfidf_vectorizer.pkl', 'rb') as f:
    tfidf = pickle.load(f)

def check_news(text):
    input_tfidf = tfidf.transform([text])
    prediction = model.predict(input_tfidf)[0]
    probability = model.predict_proba(input_tfidf)[0]

    if prediction == 1:
        return f"✅ REAL NEWS — {probability[1]*100:.2f}% confidence"
    else:
        return f"🚨 FAKE NEWS — {probability[0]*100:.2f}% confidence"

app = gr.Interface(
    fn=check_news,
    inputs=gr.Textbox(lines=5, placeholder="Paste any news headline or article here..."),
    outputs=gr.Textbox(label="Result"),
    title="📰 Fake News Detector",
    description="Enter a news headline or article to check if it is Real or Fake!",
    examples=[
        ["As U.S. budget fight looms, Republicans flip their fiscal script"],
        ["Donald Trump Sends Out Embarrassing New Year Eve Message This is Disturbing"]
    ]
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://007296a3555953ea10.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
from google.colab import files
files.download('fake_news_app.py')
files.download('fake_news_model.pkl')
files.download('tfidf_vectorizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>